###Data Cleaning Summary

####Transformations Performed

- Converted order date/time columns from STRING to TIMESTAMP.
- Converted shipping_limit_date from STRING to TIMESTAMP.
- Replaced NULL product categories with `Unknown`.
- Trimmed customer city and state values.
- Preserved valid NULL delivery timestamps because they represent missing source data.
- Preserved `not_defined` payment records for traceability.

###Validation

All cleaned tables were validated against the raw table row counts.

- Customers: 99,441
- Orders: 99,441
- Order Items: 112,650
- Order Payments: 103,886
- Products: 32,951

No records were lost during the cleaning process.

In [0]:
%sql

DESCRIBE raw_orders;

In [0]:
%sql
DESCRIBE raw_customers;

In [0]:
%sql
DESCRIBE raw_order_items;

In [0]:
%sql
DESCRIBE raw_order_payments;

In [0]:
%sql
DESCRIBE raw_products;

#####clean orders

In [0]:
%sql

CREATE OR REPLACE TABLE clean_orders AS
SELECT
    order_id,
    customer_id,
    order_status,

    TO_TIMESTAMP(order_purchase_timestamp, 'dd-MM-yyyy HH:mm') AS order_purchase_timestamp,

    TO_TIMESTAMP(order_approved_at, 'dd-MM-yyyy HH:mm') AS order_approved_at,

    TO_TIMESTAMP(order_delivered_carrier_date, 'dd-MM-yyyy HH:mm') AS order_delivered_carrier_date,

    TO_TIMESTAMP(order_delivered_customer_date, 'dd-MM-yyyy HH:mm') AS order_delivered_customer_date,

    TO_TIMESTAMP(order_estimated_delivery_date, 'dd-MM-yyyy HH:mm') AS order_estimated_delivery_date

FROM raw_orders;

In [0]:
%sql
DESCRIBE clean_orders

#####Validate the conversion

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN order_purchase_timestamp IS NULL THEN 1 ELSE 0 END) AS null_purchase,
    SUM(CASE WHEN order_approved_at IS NULL THEN 1 ELSE 0 END) AS null_approved,
    SUM(CASE WHEN order_delivered_carrier_date IS NULL THEN 1 ELSE 0 END) AS null_carrier,
    SUM(CASE WHEN order_delivered_customer_date IS NULL THEN 1 ELSE 0 END) AS null_delivered,
    SUM(CASE WHEN order_estimated_delivery_date IS NULL THEN 1 ELSE 0 END) AS null_estimated
FROM clean_orders;

#####Clean order_items

In [0]:
%sql

CREATE OR REPLACE TABLE clean_order_items AS
SELECT
    order_id,
    order_item_id,
    product_id,
    seller_id,

    TO_TIMESTAMP(
        shipping_limit_date,
        'dd-MM-yyyy HH:mm'
    ) AS shipping_limit_date,

    price,
    freight_value

FROM raw_order_items;

In [0]:
%sql
describe clean_order_items

#####Validate the timestamp conversion

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN shipping_limit_date IS NULL THEN 1 ELSE 0 END) AS null_shipping_limit
FROM clean_order_items;

#####Check raw_order_payments

In [0]:
%sql

SELECT *
FROM raw_order_payments
WHERE payment_type = 'not_defined';

####clean_order_payments

In [0]:
%sql

CREATE OR REPLACE TABLE clean_order_payments AS
SELECT
    order_id,
    payment_sequential,
    payment_type,
    payment_installments,
    payment_value
FROM raw_order_payments;

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT payment_type) AS distinct_payment_types,
    SUM(CASE WHEN payment_type IS NULL THEN 1 ELSE 0 END) AS null_payment_type
FROM clean_order_payments;

#####Cleaning the Products table

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    SUM(CASE
        WHEN product_category_name IS NULL
             OR TRIM(product_category_name) = ''
        THEN 1 ELSE 0
    END) AS missing_category
FROM raw_products;

In [0]:
%sql

SELECT
    product_category_name,
    COUNT(*) AS product_count
FROM raw_products
WHERE product_category_name IS NULL
   OR TRIM(product_category_name) = ''
GROUP BY product_category_name;

####clean_products

In [0]:
%sql

CREATE OR REPLACE TABLE clean_products AS
SELECT
    product_id,

    CASE
        WHEN product_category_name IS NULL THEN 'Unknown'
        ELSE TRIM(product_category_name)
    END AS product_category_name,

    product_name_lenght,
    product_description_lenght,
    product_photos_qty,
    product_weight_g,
    product_length_cm,
    product_height_cm,
    product_width_cm

FROM raw_products;

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    SUM(CASE
        WHEN product_category_name IS NULL THEN 1
        ELSE 0
    END) AS null_categories,
    SUM(CASE
        WHEN product_category_name = 'Unknown' THEN 1
        ELSE 0
    END) AS unknown_categories
FROM clean_products;

#####clean_customers

In [0]:
%sql

CREATE OR REPLACE TABLE clean_customers AS
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix,
    TRIM(customer_city) AS customer_city,
    TRIM(customer_state) AS customer_state
FROM raw_customers;

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT customer_id) AS unique_customer_ids,
    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_customer_ids
FROM clean_customers;

#####Final cleaning validation

In [0]:
%sql

SELECT 'customers' AS table_name, COUNT(*) AS row_count
FROM clean_customers

UNION ALL

SELECT 'orders', COUNT(*)
FROM clean_orders

UNION ALL

SELECT 'order_items', COUNT(*)
FROM clean_order_items

UNION ALL

SELECT 'order_payments', COUNT(*)
FROM clean_order_payments

UNION ALL

SELECT 'products', COUNT(*)
FROM clean_products;